# 02 — Corpus Analysis | Updated CITI-Senti Corpus

**Project:** English–Tagalog–Ilokano multilingual citizen feedback / aspect-based sentiment analysis (ABSA)  
**Source:** `citizen_feedback_raw_v1 (2).xlsx`  
**Authoritative input worksheets:** `preprocessed_corpus` and `preprocessed_aspect_annotations` **only**.

This notebook reproduces the descriptive corpus analysis from the team's previous `02_corpus_analysis.ipynb` using the new workbook. It keeps two denominators separate: **unique comments** and **target-level aspect annotations**. It never modifies the source Excel workbook.

**Provenance:** The uploaded revision includes 973 original/collected comments (IDs through `C1131`) and 4,000 generated comments (IDs `C1132`–`C5131`). Synthetic comments have simulated platform metadata; they are **not evidence of public posts**. Generated aspect/sentiment annotations may be draft labels and require human quality control before training. Provenance is analyzed separately rather than hidden.

## 1. Setup and read the two authoritative worksheets

Use `pandas`, `numpy`, `matplotlib`, and `openpyxl` in your notebook environment. If the workbook is not in the working folder or its parent, set `DATA_FILE` to its actual path.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

FILENAME = 'citizen_feedback_raw_v1 (2).xlsx'
CANDIDATES = [Path(FILENAME), Path('..') / FILENAME, Path('/mnt/data') / FILENAME]
DATA_FILE = next((p for p in CANDIDATES if p.is_file()), None)
if DATA_FILE is None:
    raise FileNotFoundError(f'Locate {FILENAME} and update DATA_FILE in this cell.')

OUTPUT_DIR = Path('outputs') / 'corpus_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

corpus_all = pd.read_excel(DATA_FILE, sheet_name='preprocessed_corpus', dtype=str, keep_default_na=False)
aspects_all = pd.read_excel(DATA_FILE, sheet_name='preprocessed_aspect_annotations', dtype=str, keep_default_na=False)
# Ignore entirely blank Excel formatting columns, e.g. 'Unnamed: 16'.
aspects_all = aspects_all.loc[:, ~aspects_all.columns.astype(str).str.startswith('Unnamed:')]
print('Workbook:', DATA_FILE.resolve())
print('Preprocessed corpus rows:', len(corpus_all), '| Aspect annotation rows:', len(aspects_all))
print('Corpus columns:', list(corpus_all.columns))
print('Aspect columns:', list(aspects_all.columns))

## 2. Build the analysis tables and identify generated versus original records

Rows with invalid/blank comment IDs are excluded. **Provenance comes directly from `preprocessed_corpus.source_type`:** `synthetic` indicates a generated comment; platform names identify original/collected comments. The former `C1132+` boundary is shown only as a consistency diagnostic; `source_reference` is not used to classify provenance. Ensure the two authoritative sheets have matching source metadata.

In [ ]:
COMMENT_RE = r'^C\d{4,}$'
SYNTHETIC_MIN_ID = 1132
SYNTHETIC_REF_PREFIXES = ('SIMULATED', 'SYNTHETIC')

valid_id = lambda s: s.astype(str).str.strip().str.fullmatch(COMMENT_RE, na=False)
corpus = corpus_all.loc[valid_id(corpus_all['comment_id'])].copy().reset_index(drop=True)
aspects = aspects_all.loc[valid_id(aspects_all['comment_id'])].copy().reset_index(drop=True)

for frame in (corpus, aspects):
    frame['comment_id'] = frame['comment_id'].str.strip()

# source_type is now the authoritative provenance field: 'synthetic' vs actual platforms.
source_labels = corpus['source_type'].astype(str).str.strip().str.casefold()
valid_sources = {'facebook', 'tiktok', 'threads', 'x', 'reddit', 'synthetic'}
unknown_sources = sorted(set(source_labels) - valid_sources)
assert not unknown_sources, f'Unexpected or empty source_type values: {unknown_sources}'
corpus['_origin'] = np.where(source_labels.eq('synthetic'), 'Synthetic', 'Original / collected')
corpus['_id_number'] = corpus['comment_id'].str[1:].astype(int)
# Historical ID cutoff is only a diagnostic, not the mechanism for assigning provenance.
id_mismatches = (corpus['_id_number'].ge(SYNTHETIC_MIN_ID) != source_labels.eq('synthetic')).sum()
if id_mismatches:
    print(f'NOTE: {id_mismatches} rows differ from the old C1132+ ID rule; source_type takes precedence.')

# Audit annotation-sheet metadata before analysis. Analysis derives platform from corpus.
expected_sources = aspects['comment_id'].map(corpus.set_index('comment_id')['source_type'])
source_mismatches = (aspects['source_type'].astype(str).str.strip().str.casefold() !=
                     expected_sources.astype(str).str.strip().str.casefold()).sum()
if source_mismatches:
    print(f'WARNING: {source_mismatches} annotation rows have outdated source_type; update that sheet too.')
corpus['_language'] = (corpus.get('primary_language_canonical', corpus['primary_language'])
                       .replace({'Ilocano': 'Ilokano'}).replace('', 'Unspecified'))
corpus['_switch'] = corpus.get('code_switched_canonical', corpus['code_switched']).replace('', 'Unspecified')
corpus['_source'] = corpus['source_type'].str.strip().replace({'Tiktok': 'TikTok', 'TIKTOK': 'TikTok', 'synthetic': 'Synthetic'})
corpus['_topic'] = corpus['source_topic'].replace('', 'Unspecified')
corpus['_policy'] = corpus['legislation_or_policy'].replace('', 'Unspecified')
corpus['_char_length'] = corpus['model_text'].str.len()
corpus['_token_count'] = pd.to_numeric(corpus['token_count_full'], errors='coerce')
corpus['_token_count_filtered'] = pd.to_numeric(corpus['token_count_filtered'], errors='coerce')

aspects['_aspect_code'] = aspects['aspect_code'].str.extract(r'(A\d{2})', expand=False)
aspects['_aspect_label'] = aspects['aspect_category'].str.replace(r'^A\d{2}\s*[—–-]\s*', '', regex=True)
aspects['_sentiment'] = aspects['sentiment'].str.strip().str.title()
aspects['_origin'] = aspects['comment_id'].map(corpus.set_index('comment_id')['_origin'])
aspects['_language'] = aspects['comment_id'].map(corpus.set_index('comment_id')['_language'])
aspects['_switch'] = aspects['comment_id'].map(corpus.set_index('comment_id')['_switch'])
aspects['_source'] = aspects['comment_id'].map(corpus.set_index('comment_id')['_source'])
aspects['_topic'] = aspects['comment_id'].map(corpus.set_index('comment_id')['_topic'])

print('Valid unique comments:', corpus['comment_id'].nunique())
print('Valid annotation rows:', len(aspects))
display(corpus['_origin'].value_counts().rename_axis('Provenance').reset_index(name='Comments'))

## 3. Data integrity and annotation coverage

These checks do not silently relabel, deduplicate, or discard valid annotations. Each comment may have multiple target-level annotation rows, including repeated categories for **different** targets.

In [ ]:
corpus_ids = set(corpus['comment_id'])
annotation_ids = set(aspects['comment_id'])
annotation_count = aspects.groupby('comment_id').size()
reported_count = pd.to_numeric(corpus['aspect_annotation_count'], errors='coerce')
actual_count = corpus['comment_id'].map(annotation_count).fillna(0).astype(int)

integrity = pd.DataFrame([
    ('Unique comment IDs', corpus['comment_id'].is_unique),
    ('Unique aspect IDs', aspects['aspect_id'].is_unique),
    ('Every aspect links to a corpus comment', annotation_ids.issubset(corpus_ids)),
    ('Every included corpus comment is annotated', corpus_ids.issubset(annotation_ids)),
    ('All model_text values populated', corpus['model_text'].str.strip().ne('').all()),
    ('Every aspect has valid A01–A11 code', aspects['_aspect_code'].isin([f'A{i:02d}' for i in range(1,12)]).all()),
    ('All sentiments are one of three classes', aspects['_sentiment'].isin(['Negative','Neutral','Positive']).all()),
    ('Source annotation counts equal actual rows', (reported_count == actual_count).all()),
    ('No blank opinion-target expression', aspects['aspect_text'].str.strip().ne('').all()),
], columns=['Check','Passed'])
display(integrity)
hard_checks = integrity.loc[integrity['Check'].ne('Source annotation counts equal actual rows')]
assert hard_checks['Passed'].all(), 'Review failed core integrity checks before splitting.'
count_mismatch = corpus.loc[reported_count.ne(actual_count), ['comment_id', 'aspect_annotation_count']].copy()
count_mismatch['actual_annotation_rows'] = actual_count.loc[count_mismatch.index].to_numpy()
print('NOTE: stale aspect_annotation_count values:', len(count_mismatch))
if len(count_mismatch):
    display(count_mismatch.head(15))

inventory = pd.DataFrame([
    ('Comments', len(corpus)), ('Aspect annotations', len(aspects)),
    ('Original/collected comments', int(corpus['_origin'].eq('Original / collected').sum())),
    ('Synthetic comments', int(corpus['_origin'].eq('Synthetic').sum())),
    ('Unique comment IDs with annotations', aspects['comment_id'].nunique()),
    ('Comments with >1 annotation', int(annotation_count.gt(1).sum())),
    ('Multi-aspect (distinct-category) comments', int(aspects.groupby('comment_id')['_aspect_code'].nunique().gt(1).sum())),
], columns=['Measure','Count'])
display(inventory)

mismatched_comment_text = (aspects.assign(_expected=aspects['comment_id'].map(corpus.set_index('comment_id')['model_text']))
                           .query('model_text != _expected'))
print('Aspect rows with model_text not matching corpus:', len(mismatched_comment_text))
assert mismatched_comment_text.empty, 'Annotation and corpus model_text disagree; reconcile before splitting.' 

## 4. Original vs synthetic composition

The corpus now records generated comments with `source_type = synthetic`, instead of assigning a fabricated social-media platform. Platform statistics for actual social-media sources use the original/collected subset only.

In [ ]:
def count_table(values, label):
    tab = values.fillna('Unspecified').astype(str).value_counts(dropna=False).rename_axis(label).reset_index(name='Count')
    tab['Percentage'] = (100 * tab['Count'] / max(tab['Count'].sum(),1)).round(2)
    return tab

origin_dist = count_table(corpus['_origin'], 'Provenance')
origin_by_language = pd.crosstab(corpus['_origin'], corpus['_language'])
origin_by_source = pd.crosstab(corpus['_origin'], corpus['_source'])
display(origin_dist, origin_by_language, origin_by_source)
fig, ax = plt.subplots(figsize=(6,3.5))
ax.bar(origin_dist['Provenance'], origin_dist['Count'])
ax.set(title='Corpus size by provenance', ylabel='Unique comments')
plt.xticks(rotation=10)
plt.tight_layout(); plt.show()

## 5. Language composition and code-switching

**Denominator: unique comments.** Canonicalize `Ilocano` to `Ilokano` for analysis only. Keep mixed-language comments as their own primary-language group.

In [ ]:
language_dist = count_table(corpus['_language'], 'Primary language')
switch_dist = count_table(corpus['_switch'], 'Code-switched')
language_by_origin = pd.crosstab(corpus['_language'], corpus['_origin'])
combination_dist = count_table(corpus['languages'].str.replace('Ilocano','Ilokano',regex=False), 'Languages present')
display(language_dist, switch_dist, combination_dist.head(20), language_by_origin)
ax=language_by_origin.plot.bar(figsize=(8,4), rot=0)
ax.set(ylabel='Comments', title='Primary language by provenance')
plt.tight_layout(); plt.show()

## 6. Length, tokens, and lexical diversity

These are the **preprocessing tokenizer** counts, not GLOT500-m SentencePiece/subword counts. Model inputs remain `model_text`; no filtered-token representation replaces it.

In [ ]:
length_summary = corpus[['_char_length','_token_count','_token_count_filtered']].describe(
    percentiles=[.25,.50,.75,.90,.95,.99]
).round(2)
length_summary.columns = ['Characters (model_text)','Full preprocessing tokens','Filtered preprocessing tokens']
display(length_summary)
lexical_tokens = [t for x in corpus['tokenized_text_filtered'].fillna('') for t in x.split() if t]
token_counts = Counter(lexical_tokens)
lexical_summary = pd.DataFrame([
    ('Filtered token occurrences',len(lexical_tokens)),
    ('Filtered vocabulary size',len(token_counts)),
    ('Vocabulary / token occurrences',round(len(token_counts)/max(len(lexical_tokens),1),4)),
], columns=['Measure','Value'])
display(lexical_summary)
fig,ax=plt.subplots(figsize=(8,4))
ax.hist(corpus['_token_count'].dropna(),bins=35)
ax.set(xlabel='Full token count (preprocessing)',ylabel='Comments',title='Comment length distribution')
plt.tight_layout();plt.show()

## 7. Most frequent words and bigrams

Uses the existing `tokenized_text_filtered` field. Bigrams are computed **within comments** to avoid crossing comment boundaries.

In [ ]:
word_top = pd.DataFrame(token_counts.most_common(25), columns=['Token','Frequency'])
bigrams=Counter()
for line in corpus['tokenized_text_filtered'].fillna(''):
    words=line.split()
    bigrams.update(zip(words,words[1:]))
bigram_top = pd.DataFrame([(' '.join(k),v) for k,v in bigrams.most_common(25)], columns=['Bigram','Frequency'])
display(word_top,bigram_top)

## 8. Source platform, policy topic and legislation coverage

For inferential descriptions of actual collected comments, use only the original/collected subset. Synthetic platform assignments are shown separately in Section 4.

In [ ]:
original_only = corpus[corpus['_origin'].eq('Original / collected')]
platform_dist = count_table(original_only['_source'], 'Reported platform (original subset)')
topic_dist = count_table(corpus['_topic'], 'Source topic (all comments)')
policy_dist = count_table(corpus['_policy'], 'Legislation / policy (all comments)')
display(platform_dist, topic_dist, policy_dist.head(30))

## 9. Annotation density and mixed polarities

In [ ]:
aspect_per_comment = aspects.groupby('comment_id').size()
density_dist = count_table(aspect_per_comment.astype(str), 'Targets per comment')
distinct_categories = aspects.groupby('comment_id')['_aspect_code'].nunique()
contrasting = aspects.groupby('comment_id')['_sentiment'].nunique().gt(1)
annotation_density_by_origin = aspects.groupby('_origin').agg(
    annotation_rows=('aspect_id','size'), comments=('comment_id','nunique'))
annotation_density_by_origin['annotations_per_comment'] = (
    annotation_density_by_origin['annotation_rows'] / annotation_density_by_origin['comments'])
display(density_dist, annotation_density_by_origin)
print('Comments with contrasting target sentiments:',int(contrasting.sum()))
print('Comments with more than one category:',int(distinct_categories.gt(1).sum()))

## 10. Sentiment balance at the annotation level

The denominator is **aspect-annotation rows**, not comments; synthetic annotations are included but flagged by provenance.

In [ ]:
sentiment_dist = count_table(aspects['_sentiment'], 'Sentiment')
sentiment_by_origin = pd.crosstab(aspects['_origin'], aspects['_sentiment'])
display(sentiment_dist, sentiment_by_origin)
ax=sentiment_by_origin.plot(kind='bar', figsize=(7,4), rot=0)
ax.set(ylabel='Target-level annotations',title='Aspect sentiment by provenance')
plt.tight_layout();plt.show()

## 11. Aspect-category coverage and sentiment within each category

In [ ]:
aspect_dist = aspects.groupby(['_aspect_code','_aspect_label']).size().rename('Count').reset_index()
aspect_dist = aspect_dist.sort_values('_aspect_code')
aspect_dist['Percentage'] = (100*aspect_dist['Count']/len(aspects)).round(2)
category_sentiment = pd.crosstab(aspects['_aspect_code'],aspects['_sentiment']).reindex(
    index=[f'A{i:02}' for i in range(1,12)], columns=['Negative','Neutral','Positive'], fill_value=0)
category_sentiment_pct = category_sentiment.div(category_sentiment.sum(axis=1).replace(0,np.nan),axis=0).mul(100).fillna(0).round(1)
display(aspect_dist, category_sentiment, category_sentiment_pct)
fig,ax=plt.subplots(figsize=(8,5))
im=ax.imshow(category_sentiment_pct.to_numpy(),aspect='auto',vmin=0,vmax=100)
ax.set(xticks=range(3),xticklabels=category_sentiment_pct.columns,yticks=range(11),yticklabels=category_sentiment_pct.index,title='Sentiment share within each aspect (%)')
plt.colorbar(im,ax=ax,label='% of category annotations')
plt.tight_layout();plt.show()

## 12. Sentiment by primary language and code-switching

In [ ]:
sentiment_order=['Negative','Neutral','Positive']
language_sentiment=pd.crosstab(aspects['_language'],aspects['_sentiment']).reindex(columns=sentiment_order, fill_value=0)
switch_sentiment=pd.crosstab(aspects['_switch'],aspects['_sentiment']).reindex(columns=sentiment_order, fill_value=0)
language_sentiment_pct=language_sentiment.div(language_sentiment.sum(axis=1).replace(0,np.nan),axis=0).mul(100).fillna(0).round(1)
display(language_sentiment,language_sentiment_pct,switch_sentiment)

## 13. Platform/topic sentiment and duplicate risk

`source_type` identifies synthetic comments explicitly. The platform-specific sentiment table below uses original/collected annotations only; generated comments remain separately identifiable in the provenance summaries. This section also flags repeated texts and IDs; near-duplicate grouping is handled by the splitting notebook.

In [ ]:
real_aspects = aspects.loc[aspects['_origin'].eq('Original / collected')]
source_sentiment=pd.crosstab(real_aspects['_source'],real_aspects['_sentiment']).reindex(columns=sentiment_order,fill_value=0)
topic_sentiment=pd.crosstab(aspects['_topic'],aspects['_sentiment']).reindex(columns=sentiment_order,fill_value=0)
normalized_text=corpus['model_text'].map(lambda t:' '.join(unicodedata.normalize('NFC',str(t)).casefold().split()))
duplicates_summary=pd.DataFrame([
    ('Repeated normalized texts',int(normalized_text.duplicated().sum())),
    ('Repeated text_group_id values',int(corpus['text_group_id'].duplicated().sum())),
    ('Repeated aspect IDs',int(aspects['aspect_id'].duplicated().sum())),
],columns=['Check','Count'])
display(source_sentiment,topic_sentiment,duplicates_summary)

## 14. Report-ready inventory and export tables

All summaries are computed from the updated workbook, not hard-coded from the old 973-comment version. Exports are **derived analysis tables only**; source rows and annotations remain unchanged.

In [ ]:
report_summary = {
    'workbook': str(DATA_FILE),
    'comments_total': int(len(corpus)),
    'comments_original_collected': int((corpus['_origin']=='Original / collected').sum()),
    'comments_synthetic': int((corpus['_origin']=='Synthetic').sum()),
    'annotations_total': int(len(aspects)),
    'annotations_original_collected': int((aspects['_origin']=='Original / collected').sum()),
    'annotations_synthetic': int((aspects['_origin']=='Synthetic').sum()),
    'annotation_coverage': round(aspects['comment_id'].nunique()/len(corpus),4),
    'all_integrity_checks_passed': bool(integrity['Passed'].all()),
    'core_integrity_checks_passed': bool(hard_checks['Passed'].all()),
    'stale_annotation_count_rows': int(len(count_mismatch)),
    'annotation_source_type_mismatches': int(source_mismatches),
    'provenance_rule': 'preprocessed_corpus.source_type == synthetic marks generated comments; real platforms mark collected comments',
    'caveat': 'Synthetic posts and suggested labels are not verified social media observations.',
}
print(json.dumps(report_summary,indent=2,ensure_ascii=False))
tables = {
    'inventory':inventory,'integrity_checks':integrity,'annotation_count_discrepancies':count_mismatch,'provenance':origin_dist,
    'language':language_dist,'code_switching':switch_dist,
    'language_by_origin':language_by_origin,'platform_original':platform_dist,
    'topics':topic_dist,'policies':policy_dist,'text_lengths':length_summary.reset_index(),
    'lexical_summary':lexical_summary,'top_words':word_top,'top_bigrams':bigram_top,
    'annotations_per_comment':density_dist,'annotation_density_by_origin':annotation_density_by_origin.reset_index(),
    'sentiment':sentiment_dist,'sentiment_by_origin':sentiment_by_origin.reset_index(),
    'aspect_categories':aspect_dist,'category_sentiment':category_sentiment.reset_index(),
    'category_sentiment_pct':category_sentiment_pct.reset_index(),
    'language_sentiment':language_sentiment.reset_index(),
    'language_sentiment_pct':language_sentiment_pct.reset_index(),
    'switch_sentiment':switch_sentiment.reset_index(),
    'source_sentiment':source_sentiment.reset_index(),
    'topic_sentiment':topic_sentiment.reset_index(),
    'duplicates':duplicates_summary,
}
for name,frame in tables.items():
    frame.to_csv(OUTPUT_DIR/f'{name}.csv',index=False,encoding='utf-8-sig')
(OUTPUT_DIR/'corpus_report_summary.json').write_text(json.dumps(report_summary,indent=2,ensure_ascii=False),encoding='utf-8')
print('Exported',len(tables),'CSV tables and report summary to',OUTPUT_DIR.resolve())